# Agent 5 Tracking Expert 워크플로우 테스트

이 노트북은 `Tracking Expert` (트래킹 전문가)의 독립적인 워크플로우를 테스트합니다.

## 주요 기능
- 테스트 이미지 로드
- Tracking Expert 그래프 실행 (Step 1 -> Step 2 -> Step 3)
- 결과 분석 및 리포트 출력

In [1]:
# 1. 초기 설정 및 라이브러리 임포트
import sys
import os
from pathlib import Path
import json
import base64

# 프로젝트 루트 경로 설정
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(project_root))

# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print(f"✓ 프로젝트 루트: {project_root}")

✓ 프로젝트 루트: c:\Users\loidn\Documents\Projects\P_04_Scope


In [2]:
# 2. 테스트 이미지 준비
from src.utils import find_data_directory

try:
    data_dir = find_data_directory()
    test_image_name = "IMG_8113.jpg"
    test_image_path = Path(data_dir) / test_image_name
    
    if not test_image_path.exists():
        # 이미지가 없으면 첫 번째 가능한 이미지 사용
        image_files = list(Path(data_dir).glob("*.png")) + list(Path(data_dir).glob("*.jpg"))
        if image_files:
            test_image_path = image_files[0]
            print(f"⚠️ {test_image_name}을 찾을 수 없어 {test_image_path.name}을 사용합니다.")
        else:
            raise FileNotFoundError("테스트할 이미지가 없습니다.")
            
    print(f"✓ 테스트 이미지: {test_image_path}")
except Exception as e:
    print(f"❌ 오류: {e}")

✓ 테스트 이미지: c:\Users\loidn\Documents\Projects\P_04_Scope\data\IMG_8113.jpg


In [3]:
# 3. Payload 생성 함수
def create_payload(image_path):
    with open(image_path, 'rb') as f:
        image_data = f.read()
    
    ext = Path(image_path).suffix.lower()
    mime_type = 'image/png' if ext == '.png' else 'image/jpeg'
    image_base64 = base64.b64encode(image_data).decode('utf-8')
    
    return [
        {"text": "이미지를 분석하세요."},
        {"inline_data": {"mime_type": mime_type, "data": image_base64}}
    ]

In [ ]:
# [프롬프트 테스트 및 수정]
# 이 셀에서 프롬프트 내용을 직접 수정하여 테스트할 수 있습니다.
# 원본 파일: src/prompts/tracking_expert_prompts.py
# 수정 후 이 셀을 실행하면 변경된 프롬프트가 다음 단계에 적용됩니다.

import src.prompts.tracking_expert_prompts as prompt_module

print("🔄 프롬프트 오버라이드 적용 중...")

# === Step 1 ===
GET_STEP1_REACT_PROMPT_TEMPLATE = """당신은 전기 표면 방전 및 트래킹 현상 분석 전문가입니다. 주어진 이미지에서 수지상 도전로 패턴을 분석하세요.

**대상 이미지 경로:** "{image_path}"

**[도구 사용 및 운영 원칙]**
1. **입력 데이터 준수:** 도구 사용 시 `image_path` 인자는 절대 변경하지 말고 위 경로를 그대로 사용하십시오.
2. **패턴 식별을 위한 보정:**
   - 검은색 탄화 흔적이 배경(어두운 플라스틱 등)과 대비되어 잘 보여야 합니다.
   - 이미지가 너무 어둡거나 대비가 낮아 패턴 식별이 어렵다면 **`apply_clahe_filter` 도구를 호출**하십시오.
   - 흐릿하다면 `enhance_image`를 사용하십시오.

**[단계별 분석 프로세스 (Chain of Thought)]**
도구를 사용할 필요가 없거나 보정이 완료되었다면, 다음 순서대로 생각하고 분석하십시오.

1단계: 시각적 요소 추출
- 이미지 전체를 스캔하여 검은색 탄화 흔적의 분포를 관찰하세요.
- 탄화 흔적이 가지처럼 뻗어나가는 패턴을 객관적으로 식별하세요.
- 두 개의 전극(도체, 단자 등) 사이를 연결하는 경로를 찾으세요.

2단계: 특징 서술
- 발견된 패턴을 정확히 서술하세요:
  * 수지상(Dendritic) 패턴: 나뭇가지처럼 뻗어나가는 형태
  * 선형(Linear) 패턴: 직선으로 연결된 형태
  * 복잡한(Complex) 패턴: 여러 경로가 교차하거나 분기하는 형태
- 두 전극을 연결하는 경로가 있는지 확인하세요.
- 패턴의 복잡도(simple, moderate, complex)를 서술하세요.

3단계: 논리적 추론
- 트래킹은 두 전극 사이에 도전로를 형성하는 현상입니다.
- 수지상 패턴은 트래킹의 전형적인 특징이지만, 단순히 나뭇가지 모양만으로는 트래킹이라고 판단할 수 없습니다.
- **중요**: 단순 연소 흔적이나 오염도 나뭇가지 모양으로 보일 수 있습니다.
- 트래킹으로 판단하려면 반드시 **두 전극을 연결하는 경로**가 명확히 확인되어야 합니다.
- 전극 연결이 확인되지 않으면, 단순 그을음이나 화염 흔적일 가능성이 높으므로 보수적으로 판단하세요.
- 관찰된 패턴을 종합하여 트래킹 여부를 논리적으로 판단하세요.

**[출력 형식]**
모든 분석이 완료되면(또는 도구 사용이 끝난 후), 반드시 다음 JSON 형식으로 응답하세요:
{{
    "dendritic_pattern_detected": true/false,
    "pattern_type": "dendritic" | "linear" | "complex" | "none" | "unknown",
    "pattern_description": "패턴에 대한 상세 설명",
    "electrode_connection": true/false,
    "connection_description": "두 전극을 연결하는 경로 설명",
    "pattern_complexity": "simple" | "moderate" | "complex" | "unknown",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}}
"""

def get_step1_react_prompt_override(image_path: str = None) -> str:
    if image_path is None:
        return GET_STEP1_REACT_PROMPT_TEMPLATE
    return GET_STEP1_REACT_PROMPT_TEMPLATE.format(image_path=image_path)

# 함수 교체
prompt_module.get_step1_react_prompt = get_step1_react_prompt_override
print("✓ Step 1 프롬프트 오버라이드 완료")

# === Step 2 ===
GET_STEP2_REACT_PROMPT_TEMPLATE = """당신은 전기 표면 방전 및 트래킹 현상 분석 전문가입니다. 주어진 이미지에서 탄화 흔적의 광택을 분석하여 흑연화(Graphitization) 여부를 판단하세요.

**대상 이미지 경로:** "{image_path}"

**[도구 사용 및 운영 원칙]**
1. **입력 데이터 준수:** 도구 사용 시 `image_path` 인자는 절대 변경하지 말고 위 경로를 그대로 사용하십시오.
2. **광택 구분을 위한 보정:**
   - 흑연의 미세한 반짝임(Sparkle)과 단순 조명 반사(Glare)를 구분해야 합니다.
   - 이미지가 너무 밝아서 전체가 하얗게 뜨거나(Overexposed), 반대로 너무 어두워 광택이 안 보인다면 **`enhance_image` (밝기/대비 보정) 도구를 호출**하십시오.

**[치명적 주의사항 - 빛 반사와 흑연 광택 구별]**
- 사진 촬영 시 사용된 '카메라 플래시'나 '조명'에 의한 하이라이트(Spotlight)를 흑연 광택으로 오인하지 마세요.
- 흑연 광택은 탄화된 '경로(Path)'를 따라 선형으로 은은하게 나타나는 연속적인 광택입니다.
- 만약 반짝임이 이미지의 특정 한 지점에만 둥글게 맺혀 있다면, 이는 단순 조명 반사(Glare)일 가능성이 높으므로 'False'로 판단하세요.
- 젖은 표면이나 기름에 의한 반사도 광택처럼 보일 수 있으므로, 탄화 경로와의 위치 관계를 정확히 확인하세요.

**[단계별 분석 프로세스 (Chain of Thought)]**
도구를 사용할 필요가 없거나 보정이 완료되었다면, 다음 순서대로 생각하고 분석하십시오.

1단계: 광택의 위치와 분포 분석
- 먼저 반짝임이 어디에 있는지 정확히 관찰하세요.
- 반짝임이 탄화 경로를 따라 선형으로 분포하는지, 아니면 특정 지점에만 집중되어 있는지 확인하세요.
- 카메라 플래시나 조명에 의한 하이라이트는 보통 이미지의 특정 위치(중앙, 모서리 등)에 둥글게 나타납니다.

2단계: 광택의 연속성 평가
- 흑연 광택은 탄화 경로를 따라 끊김 없이 연속적으로 나타나는 경향이 있습니다.
- 조명 반사는 특정 지점(Hotspot)에만 강하게 나타나며, 경로를 따라 분포하지 않습니다.
- 광택이 탄화 경로와 일치하는지, 아니면 무관한 위치에 있는지 정확히 판단하세요.

3단계: 특징 서술
- 발견된 광택을 정확히 서술하세요:
  * 금속성 광택(Metallic Luster)인지
  * 윤기(Shininess)가 있는지
  * 무광택(Matte)인지
- 광택이 탄화된 부분에만 국한되어 있는지 위치를 정확히 서술하세요.
- 광택의 분포가 선형적(Linear)인지, 점적(Spot)인지 서술하세요.

4단계: 논리적 추론
- 일반적인 화재 그을음(Amorphous Carbon)은 무광택(Matte)이며 빛을 흡수합니다.
- 트래킹에 의해 생성된 흑연(Graphite)은 결정 구조로 인해 빛을 정반사(Specular Reflection)하여 반짝입니다.
- 광택이 탄화 경로를 따라 연속적으로 나타나고, 조명 반사가 아님이 확실할 때만 트래킹 확률을 높게 설정하세요.
- 관찰된 광택 특성을 종합하여 흑연화 여부를 논리적으로 판단하세요.

**[출력 형식]**
모든 분석이 완료되면(또는 도구 사용이 끝난 후), 반드시 다음 JSON 형식으로 응답하세요:
{{
    "luster_detected": true/false,
    "luster_type": "metallic" | "shiny" | "matte" | "none" | "unknown",
    "luster_location": "광택이 관찰된 위치 설명",
    "graphitization_evidence": true/false,
    "glare_distinction": "조명 반사와 흑연 광택의 구별 설명",
    "carbon_type": "graphite" | "amorphous" | "mixed" | "unknown",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}}
"""

def get_step2_react_prompt_override(image_path: str = None) -> str:
    if image_path is None:
        return GET_STEP2_REACT_PROMPT_TEMPLATE
    return GET_STEP2_REACT_PROMPT_TEMPLATE.format(image_path=image_path)

# 함수 교체
prompt_module.get_step2_react_prompt = get_step2_react_prompt_override
print("✓ Step 2 프롬프트 오버라이드 완료")

# === Step 3 ===
GET_STEP3_REACT_PROMPT_TEMPLATE = """당신은 전기 표면 방전 및 트래킹 현상 분석 전문가입니다. 주어진 이미지에서 탄화 경로를 따른 표면 침식을 분석하세요.

**대상 이미지 경로:** "{image_path}"

**[도구 사용 및 운영 원칙]**
1. **입력 데이터 준수:** 도구 사용 시 `image_path` 인자는 절대 변경하지 말고 위 경로를 그대로 사용하십시오.
2. **입체감 식별을 위한 보정:**
   - 표면이 파였는지(Erosion) 아니면 물질이 쌓였는지(Deposit) 구분하려면 입체감이 중요합니다.
   - 음영 대비가 약해 깊이감이 안 느껴진다면 **`apply_clahe_filter` 도구를 호출**하십시오.
   - **(제약)** 도구는 **최대 1회만** 사용하십시오. 재시도하지 마십시오.

**[단계별 분석 프로세스 (Chain of Thought)]**
도구를 사용할 필요가 없거나 보정이 완료되었다면, 다음 순서대로 생각하고 분석하십시오.

1단계: 시각적 요소 추출
- 탄화 경로를 따라 절연체 표면의 손상 상태를 관찰하세요.
- 표면이 움푹 패이거나 굴착된 부분을 객관적으로 식별하세요.
- 탄화물이 표면에 얇게 증착된 것인지, 재료가 변질된 것인지 구별하세요.

2단계: 특징 서술
- 표면 침식을 정확히 서술하세요:
  * 탄화 경로를 따라 절연체 표면이 움푹 패이거나(Eroded) 굴착된 듯한 입체적 손상이 있는지
  * 침식의 깊이(shallow, moderate, deep)를 서술하세요
  * 탄화물이 표면에 얇게 증착된 그을음인지, 재료 표면이 변질되어 형성된 구조적인 트랙인지 구분하세요
- 침식 패턴이 탄화 경로와 일치하는지 서술하세요.

3단계: 논리적 추론
- 트래킹은 표면을 갉아먹으며 진행되므로, 탄화 경로를 따라 재료가 패이거나 소실된 흔적이 남습니다.
- 구조적인 트랙은 단순 그을음과 달리 재료 자체가 변질되어 형성된 것입니다.
- 표면 침식이 탄화 패턴과 일치한다면 트래킹의 강력한 증거입니다.
- 관찰된 침식 패턴을 종합하여 트래킹 여부를 논리적으로 판단하세요.

**[출력 형식]**
모든 분석이 완료되면(또는 도구 사용이 끝난 후), 반드시 다음 JSON 형식으로 응답하세요:
{{
    "surface_erosion_detected": true/false,
    "erosion_pattern": "track_following" | "general" | "none" | "unknown",
    "erosion_depth": "shallow" | "moderate" | "deep" | "unknown",
    "carbon_type": "surface_deposit" | "structural_track" | "mixed" | "unknown",
    "erosion_description": "표면 침식에 대한 상세 설명",
    "pattern_match": true/false,
    "confidence": 0-100,
    "reasoning": "판단 근거"
}}
"""

def get_step3_react_prompt_override(image_path: str = None) -> str:
    if image_path is None:
        return GET_STEP3_REACT_PROMPT_TEMPLATE
    return GET_STEP3_REACT_PROMPT_TEMPLATE.format(image_path=image_path)

# 함수 교체
prompt_module.get_step3_react_prompt = get_step3_react_prompt_override
print("✓ Step 3 프롬프트 오버라이드 완료")

print("✅ 모든 프롬프트가 재정의되었습니다. 아래 셀을 실행하여 테스트하세요.")


In [4]:
# 4. Tracking Expert 실행
from src.graphs.tracking_expert_graph import tracking_expert_wrapper_node
from src.state import InvestigationState

print("Tracking Expert 분석 시작...")

payload = create_payload(test_image_path)

# InvestigationState 모의 구성
mock_state = InvestigationState(
    payload=payload,
    expert_reports=[],
    expert_analysis_results={},
    expert_confidence_scores={},
    expert_evidence={},
    final_verdict=None,
    errors=[],
    tracking_cached_image_data=None
)

try:
    result = tracking_expert_wrapper_node(mock_state)
    
    print("\n✓ 분석 완료!")
    print("=" * 60)
    
    # 결과 출력
    reports = result.get("expert_reports", [])
    if reports:
        print(reports[0])
    else:
        print("리포트가 생성되지 않았습니다.")
        
    print("=" * 60)
    print("세부 분석 결과:")
    analysis_results = result.get("expert_analysis_results", {}).get("tracking", {})
    print(json.dumps(analysis_results, indent=2, ensure_ascii=False))
    
except Exception as e:
    print(f"❌ 실행 중 오류 발생: {e}")
    import traceback
    traceback.print_exc()

Tracking Expert 분석 시작...

==================== Agent Reasoning & Tool Execution ====================
🛠️ [Tool Call]: apply_clahe_filter (Args: {'image_path': 'C:\\Users\\loidn\\AppData\\Local\\Temp\\tmpslh2rub2.jpg'})
   └─ 📊 [Tool Output]: CLAHE 필터 적용 완료
- 이미지 크기: 1568x1176
🛠️ [Tool Call]: analyze_dendritic_pattern_internal (Args: {'image_path': 'C:\\Users\\loidn\\AppData\\Local\\Temp\\tmpslh2rub2.jpg'})
   └─ 📊 [Tool Output]: {"dendritic_pattern_detected": false, "pattern_type": "none", "pattern_description": "절연체 표면에서 나뭇가지 형태의 탄화 도전로(Carbonized path)가 식별되지 않음. 전선 피복의 국부적인 연소와 소선의 물리적 분리 현상만 관찰됨.", "electrode_connection": false, "connection_description": "분리된 도체 가닥들 사이에 전기적으로 연결된 탄화된 선형 또는 수지상 경로가 존재하지 않음.", "pattern_com...

🧠 [Thought]:
Final Answer: 제공된 이미지 분석 결과, **수지상 도전로(Dendritic Pattern) 패턴은 식별되지 않았습니다.**

분석의 주요 내용은 다음과 같습니다:

1. **패턴 식별 결과**: 절연체 표면에서 트래킹(Tracking) 현상의 전형적인 특징인 나뭇가지 형태의 탄화 도전로가 발견되지 않았습니다.
2. **관찰 사항**: 전선 피복의 국부적인 연소 흔적과 소선(금속 가닥)의 물리적인 분리 현상이 주된 특징으로 관찰되었습

In [ ]:
# 5. 분석 결과 시각화 (AI 검출 위치 표시)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io
import numpy as np

def draw_bounding_boxes(image_path, analysis_results):
    """
    분석 결과에 포함된 Bounding Box를 이미지 위에 표시합니다.
    """
    try:
        # 이미지 로드
        img = Image.open(image_path)
        original_width, original_height = img.size
        
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(img)
        ax.set_title("AI Detected Regions", fontsize=15)
        axis_off = True
        
        colors = {'step1': 'red', 'step2': 'orange', 'step3': 'blue', 'step4': 'green'}
        labels = {'step1': 'Step 1', 'step2': 'Step 2', 'step3': 'Step 3', 'step4': 'Step 4'}
        
        legend_patches = []
        
        # 각 단계별 결과 시각화
        for step_key, color in colors.items():
            step_result = analysis_results.get(step_key, {})
            bboxes = step_result.get('bboxes', [])
            
            if bboxes:
                axis_off = False
                legend_patches.append(patches.Patch(color=color, label=f"{labels[step_key]} ({len(bboxes)})"))
                
                for box in bboxes:
                    # 0-1000 정규화 좌표 -> 픽셀 좌표 변환
                    # box format: [ymin, xmin, ymax, xmax]
                    ymin, xmin, ymax, xmax = box
                    
                    x = xmin / 1000 * original_width
                    y = ymin / 1000 * original_height
                    w = (xmax - xmin) / 1000 * original_width
                    h = (ymax - ymin) / 1000 * original_height
                    
                    # 사각형 그리기
                    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)

        if legend_patches:
            ax.legend(handles=legend_patches, loc='upper right')
            
        plt.axis('off')
        plt.show()
        
        if not legend_patches:
            print("검출된 Bounding Box 정보가 없습니다.")
            
    except Exception as e:
        print(f"시각화 중 오류 발생: {e}")

# 시각화 함수 실행
if 'test_image_path' in locals() and 'result' in locals():
    # 결과에서 전문가 키 찾기 (동적)
    expert_results = result.get("expert_analysis_results", {})
    target_key = next(iter(expert_results), None) # 첫 번째 키 사용
    
    if target_key:
        print(f"Visualizing results for expert: {target_key}")
        analysis_results = expert_results[target_key]
        draw_bounding_boxes(test_image_path, analysis_results)
    else:
        print("분석 결과(expert_analysis_results)가 비어 있습니다.")
else:
    print("테스트 이미지 경로 또는 분석 결과가 없습니다.")